# Cats vs Dogs: reproduce every result in the README

Runtime → Change runtime type → **T4 GPU**. The two final models take a few hours on a T4, most of it the custom CNN (80 epochs). The optional ablation takes much longer.


In [ ]:
!git clone https://github.com/xzhou-C/Kaggle_Cat_Dog_CNN.git
%cd cats-vs-dogs
!pip install -q tabulate gradio
%env PYTHONPATH=src

## 1. Get the raw data (Microsoft Kaggle Cats and Dogs, ~800 MB)

In [ ]:
!wget -q https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -q kagglecatsanddogs_5340.zip
# Already have it on Drive? Instead: from google.colab import drive; drive.mount('/content/drive')
# and pass --src /content/drive/MyDrive/PetImages below.

## 2. Clean, de-duplicate and split 80/10/10

In [ ]:
!python -m catdog.prepare_data --src PetImages --out data

## 3. Train both models
The final from-scratch CNN (GAP head, two 3x3 convs per block, cosine LR) and EfficientNetB0 pretrained on ImageNet (head first, then fine-tune the top 40 layers).

In [ ]:
!python -m catdog.train --model cnn --out runs/cnn --head gap --convs-per-block 2 --epochs 80 --schedule cosine --patience 15

In [ ]:
!python -m catdog.train --model effnet --out runs/effnet --epochs 5 --finetune-epochs 10

## 4. Evaluate on the held-out test set and compare

In [ ]:
!python -m catdog.evaluate --run runs/cnn
!python -m catdog.evaluate --run runs/effnet
!python -m catdog.compare runs/cnn runs/effnet --out reports/results.md

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/results.md').read()))
for run in ['cnn', 'effnet']:
    display(Markdown(f'### {run}'))
    display(pd.read_csv(f'runs/{run}/eval/slices.csv'))
    for plot in ['confusion', 'roc', 'calibration', 'errors_gradcam']:
        display(Image(f'runs/{run}/eval/{plot}.png', width=500))

## 4b. Optional: CNN ablation (one change per run)
Trains the five CNN variants from the README table and writes `reports/ablation.md`.

In [ ]:
!python -m catdog.ablation --out runs/ablation --report reports/ablation.md

## 5. Save results (copy plots into reports/ for the README, and the model to Drive)

In [ ]:
!mkdir -p reports/figures
!for r in cnn effnet; do for p in confusion roc calibration errors_gradcam; do cp runs/$r/eval/$p.png reports/figures/${r}_$p.png; done; cp runs/$r/eval/slices.csv reports/${r}_slices.csv; done
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cats-vs-dogs && cp -r runs reports /content/drive/MyDrive/cats-vs-dogs/

## 6. Try the demo

In [ ]:
import app
app.build_app('runs/effnet/model.keras').launch(share=True)